# approx_exp_max_sharpe
背景：
- 当对大量策略进行回测时，即使这些策略是完全随机的，也存在一个 Sharpe Ratio 最优的策略，这纯粹是因为这个策略 “运气好”
- 需要估计 “运气成分” 的最大值，才能区分是 “运气好” 还是策略本身存在优势

从正态分布 $N\left( {\mu ,{\sigma ^2}} \right)$ 抽取 $N$ 个独立的随机变量，最大值的期望记为
$$SR\left( {\mu ,\sigma ,N} \right) = \mu  + \sigma \left[ {\left( {1 - \gamma } \right){\Phi ^{ - 1}}\left( {1 - \frac{1}{N}} \right) + \gamma {\Phi ^{ - 1}}\left( {1 - \frac{1}{{Ne}}} \right)} \right]$$
- ${\Phi ^{ - 1}}\left( p \right)$：标准正态分布的逆累积分布函数，给定 $p$，返回 $z$，使得标准正态分布中随机变量小于等于 $z$ 的概率恰好是 $p$
- $\gamma$：Euler-Mascheroni 常数
- $e$：自然常数

```python
def approx_exp_max_sharpe(mean_sharpe: float, var_sharpe: float, nb_trials: int) -> float:

    return mean_sharpe + np.sqrt(var_sharpe) * \
           ((1 - np.euler_gamma) * norm.ppf(1 - 1 / nb_trials) + np.euler_gamma * norm.ppf(1 - 1 / (nb_trials * np.e)))
```

# deflated_sharpe_ratio
考虑 $N$ 个策略，根据收益数据计算出
- 各自的夏普比 $\widehat{sr}_i = \frac{{{{\widehat \mu }_i}}}{{{{\widehat \sigma }_i}}}$，峰度 $K_i$ 和偏度 $S_i$（$i = 1, \cdots ,N$）
- 所有夏普比 $\left\{ {{{\widehat {sr}}_i}} \right\}$ 的样本方差 $\operatorname{var}$

假设：所有策略的真实夏普比相互独立，且服从正态分布 $N\left( {0,{\mathop{\rm var}} } \right)$
- 那么其中最大真实夏普比的均值应该为 $SR\left( {0,\sigma ,N} \right)$

夏普比的估计方式 $\widehat{sr} = \frac{{{\widehat \mu }}}{{{\widehat \sigma }}}$ 作为一个随机变量， 参考下文推导，其理论方差为 $\operatorname{Var}\left[ \widehat{sr} \right] =\frac{1}{T}\left(1-S \frac{\mu}{\sigma}+\left(\frac{\kappa(T-1)-(T-3)}{4(T-1)}\right)\left(\frac{\mu}{\sigma}\right)^2\right)$

那么当 $T \to \infty $ 时，应当
$$\frac{{\widehat {sr} - SR\left( {0,\sigma ,N} \right)}}{{\sqrt {\frac{1}{{T - 1}}\left( {1 - S \cdot sr + \frac{{K - 1}}{4} \cdot {{\widehat {sr}}^2}} \right)} }} \to N\left( {0,1} \right)$$

所以当 $$DSR = \Phi \left( {\frac{{\widehat {sr} - SR\left( {0,\sigma ,N} \right)}}{{\sqrt {\frac{1}{{T - 1}}\left( {1 - S \cdot sr + \frac{{K - 1}}{4} \cdot {{\widehat {sr}}^2}} \right)} }}} \right)$$ 接近于 $1$ 时，可以证明该策略本身存在优势，而非只是“运气好”。


## 源码
```python
def deflated_sharpe_ratio(*, est_sharpe: tp.Array1d, var_sharpe: float, nb_trials: int,
                          backtest_horizon: int, skew: tp.Array1d, kurtosis: tp.Array1d) -> tp.Array1d:

    SR0 = approx_exp_max_sharpe(0, var_sharpe, nb_trials)

    return norm.cdf(((est_sharpe - SR0) * np.sqrt(backtest_horizon - 1)) /
                    np.sqrt(1 - skew * est_sharpe + ((kurtosis - 1) / 4) * est_sharpe ** 2))
```

参数
- `est_sharpe` (tp.Array1d): 估计的夏普比率数组。通过历史数据计算得到的策略夏普比率
- `var_sharpe` (float): 夏普比率的方差
- `nb_trials` (int): 总共测试的策略数量
- `backtest_horizon` (int): 策略的回测时间长度
- `skew` (tp.Array1d): 各策略收益分布的偏度
  - 衡量收益分布相对于正态分布的不对称程度
  - 负偏度表示极端损失的概率较高
- `kurtosis` (tp.Array1d): 收益分布的峰度数组
  - 衡量收益分布的尾部厚度
  - 高峰度表示极端事件发生概率较高

返回：tp.Array1d，紧缩夏普比率数组
- 取值范围在[0, 1]之间
- 接近1表示策略具有高度统计显著性
- 接近0.5表示策略与随机策略无显著差异
- 低于0.5表示策略表现可能不如随机策略

## Delta 方法
### $\operatorname{Var}[g(X, Y)]$
对于 $g\left( {X,Y} \right)$，其中 $X$ 和 $Y$ 为两个随机变量
$$g(X,Y) \approx g\left( {{\mu _X},{\mu _Y}} \right) + {\left. {\frac{{\partial g}}{{\partial X}}} \right|_{\left( {{\mu _X},{\mu _Y}} \right)}}\left( {X - {\mu _X}} \right) + {\left. {\frac{{\partial g}}{{\partial Y}}} \right|_{\left( {{\mu _X},{\mu _Y}} \right)}}\left( {Y - {\mu _Y}} \right)$$
用 $g_X^{\prime}$ 和 $g_Y^{\prime}$ 来表示在 $\left(\mu_X, \mu_Y\right)$ 处对 $X$ 和 $Y$ 的偏导数，因此
$$g(X, Y) \approx g\left(\mu_X, \mu_Y\right)+g_X^{\prime}\left(X-\mu_X\right)+g_Y^{\prime}\left(Y-\mu_Y\right)$$
因此有
$$
\operatorname{Var}[g(X, Y)] \approx\left(g_X^{\prime}\right)^2 \sigma_X^2+\left(g_Y^{\prime}\right)^2 \sigma_Y^2+2 g_X^{\prime} g_Y^{\prime} \sigma_{X Y}
$$
### ${\mathop{\rm Cov}\nolimits} \left[ {{g_1}(X,Y),{g_2}(X,Y)} \right]$
可以进一步推导得到
$${\mathop{\rm Cov}\nolimits} \left[ {{g_1}(X,Y),{g_2}(X,Y)} \right] \approx g_{1,X}^\prime g_{2,X}^\prime {\mathop{\rm Var}\nolimits} [X] + g_{1,Y}^\prime g_{2,Y}^\prime {\mathop{\rm Var}\nolimits} [Y] + \left( {g_{1,X}^\prime g_{2,Y}^\prime  + g_{1,Y}^\prime g_{2,X}^\prime } \right){\mathop{\rm Cov}\nolimits} [X,Y]$$
### $\operatorname{Var}\left[ {g\left( {{X_1}, \cdots } \right)} \right]$
Delta 方法可以推广到涉及 $k$ 个随机变量的情况。假设有一个随机向量 $X =$ $\left(X_1, X_2, \ldots, X_k\right)$ ，其期望向量为 $E[ X ]= \mu =\left(\mu_1, \mu_2, \ldots, \mu_k\right)$ ，并且协方差矩阵为 $\Sigma$

有一个函数 $g( X )$，计算 $g( X )$ 在点 $\mu$ 处的偏导数（梯度向量）：
$$
\nabla g(\mu)=\left(\left.\frac{\partial g}{\partial X_1}\right|_\mu,\left.\frac{\partial g}{\partial X_2}\right|_\mu, \ldots,\left.\frac{\partial g}{\partial X_k}\right|_\mu\right)
$$
方羞的近似公式为：
$$
\operatorname{Var}[g( X )] \approx[\nabla g(\mu)]^T \Sigma [\nabla g(\mu)]
$$
其中，$[\nabla g(\mu)]^T$ 是梯度向量的转置（行向量）， $\Sigma$ 是协方差矩阵。展开形式为：
$$
\left.\left.\operatorname{Var}[g( X )] \approx \sum_{i=1}^k \sum_{j=1}^k \frac{\partial g}{\partial X_i}\right|_\mu \frac{\partial g}{\partial X_j}\right|_\mu \operatorname{Cov}\left[X_i, X_j\right]
$$
当 $i=j$ 时， $\operatorname{Cov}\left[X_i, X_i\right]=\operatorname{Var}\left[X_i\right]$ 。

## Sharpe ratio 的方差

考虑长度为 $T$ 的独立同分布序列 ${X_1}, \cdots {X_T}$，令 $\hat \mu $ 和 $\hat \sigma $ 分别表示其样本均值和标准差，则 Sharpe ratio 的方差为
$$Var\left[ {\frac{\hat \mu }{\hat \sigma }} \right] = \frac{1}{{{\sigma ^2}}}Var\left[ {\hat \mu } \right] + \frac{{{\mu ^2}}}{{{\sigma ^4}}}Var\left[ \hat \sigma  \right] - 2\frac{\mu }{{{\sigma ^3}}}Cov\left[ {\hat \mu,\hat \sigma} \right]$$
其中 $Var\left[ {\hat \mu } \right] = \frac{{{\sigma ^2}}}{T}$。

考虑 $\hat \sigma = \sqrt {{{\hat \sigma}^2}} $，其中 $E\left[{\hat \sigma}^2\right]=E\left[\frac{1}{T-1} \sum_{i=1}^T\left(X_i-\hat \mu\right)^2\right]=\sigma^2$，所以
$${\mathop{\rm Var}\nolimits} \left[ \hat \sigma \right] = {\left( {\frac{1}{{2\sigma }}} \right)^2}{\mathop{\rm Var}\nolimits} \left[ {{{\hat \sigma}^2}} \right] = \frac{1}{{4{\sigma ^2}T}}\left( {{\mu _4} - \frac{{T - 3}}{{T - 1}}{\sigma ^4}} \right)$$
- 可以证明 $\operatorname{Var}\left[{\hat \sigma}^2\right]=\frac{1}{T}\left(\mu_4-\frac{T-3}{T-1} \sigma^4\right)$，其中 $\mu_4=E\left[\left(X_i-\mu\right)^4\right]$ 是总体的四阶中心矩

考虑 $${\mathop{\rm Cov}\nolimits} [\hat \mu,\hat \sigma] \approx (1)(0){\mathop{\rm Var}\nolimits} [\hat \mu] + (0)\left( {\frac{1}{{2\sigma }}} \right){\mathop{\rm Var}\nolimits} \left[ {{{\hat \sigma}^2}} \right] + \left( {(1)\left( {\frac{1}{{2\sigma }}} \right) + (0)(0)} \right){\mathop{\rm Cov}\nolimits} \left[ {\hat \mu,{{\hat \sigma}^2}} \right]$$
- 可以证明 $\operatorname{Cov}\left[\hat \mu, {\hat \sigma}^2\right]=\frac{\mu_3}{T}$，其中 $\mu_3=E\left[\left(X_i-\mu\right)^3\right]$ 是总体的三阶中心矩。对于对称分布，$\mu_3=0$ 

因此可以得到
$$
\operatorname{Cov}[\hat \mu, \hat \sigma] \approx \frac{\mu_3}{2 T \sigma}
$$

定义**峰度**$\kappa=E\left[\left(\frac{X-\mu}{\sigma}\right)^4\right]=\frac{\mu_4}{\sigma^4}$ 和**偏度**$S=E\left[\left(\frac{X-\mu}{\sigma}\right)^3\right]=\frac{\mu_3}{\sigma^3}$，最终可以得到
\begin{equation}
\begin{aligned}
Var\left[ {\frac{\hat \mu }{\hat \sigma }} \right] & =\frac{1}{T}+\frac{\mu^2}{4 T \sigma^2}\left(\frac{\kappa(T-1)-(T-3)}{T-1}\right)-\frac{\mu S}{T \sigma} \\
& =\frac{1}{T}\left(1-S \frac{\mu}{\sigma}+\left(\frac{\kappa(T-1)-(T-3)}{4(T-1)}\right)\left(\frac{\mu}{\sigma}\right)^2\right)
\end{aligned}
\end{equation}